# 02 — Moving-Edge Responses (Whole-Brain mcHH)

Direction selectivity analysis for the **whole visual brain** network.

**Estimated time**: ~55 min for 12-direction sweep (300ms each).
Consider running with fewer directions (e.g., 4) for a quick test.

In [ ]:
import sys, time, logging
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, '../..')
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')

from neuro_framework.models.fafb_mc_network import FAFBMCNetwork
from neuro_framework.stimulus.visual import MovingEdgeStimulus, hex_grid_coords

In [ ]:
DATA_DIR  = '../../mcHH/data/visual_whole_brain'
SWC_PATH  = '../../mcHH/data/generic_2comp.swc'
ION_RULES = '../data/fafb_ion_channel_rules.csv'
SYN_RULES = '../data/fafb_synapse_rules.csv'

DT = 0.1
net = FAFBMCNetwork.from_preprocessed(
    data_dir=DATA_DIR, swc_path=SWC_PATH,
    ion_rules_path=ION_RULES, syn_rules_path=SYN_RULES,
    ncomp=2, min_syn_count=3, dt=DT,
)
print(net)

## 1. Photoreceptor coordinates

In [ ]:
n_photo = net.input_mask.sum().item()
r = 1
while 3*r*(r-1)+1 < n_photo:
    r += 1
coords = hex_grid_coords(r)[:n_photo]
coords -= coords.min(axis=0)
coords /= (coords.max(axis=0) - coords.min(axis=0) + 1e-8)
print(f'Photoreceptor coords: {coords.shape}')

## 2. Simulate moving edges

Use 4 cardinal directions for a quick test.  Change to `np.arange(0,360,30)`
for a full 12-direction sweep (~55 min).

In [ ]:
I_AMP = 30.0
T_PRE, T_STIM, T_POST = 50.0, 200.0, 50.0
SPEED = 0.03

directions = np.arange(0, 360, 90)  # 4 dirs for speed; use 30° steps for full
all_responses = {}

for d_deg in directions:
    stim = MovingEdgeStimulus(
        coords=coords, direction_deg=float(d_deg), speed=SPEED,
        amplitude=1.0, t_start=T_PRE, t_end=T_PRE+T_STIM, dt=DT,
    )
    s_arr = stim.as_numpy()
    n_pre  = int(T_PRE / DT)
    n_post = int(T_POST / DT)
    full = np.zeros((n_pre + s_arr.shape[0] + n_post, s_arr.shape[1]), dtype=np.float32)
    full[n_pre:n_pre+s_arr.shape[0]] = s_arr
    x = torch.from_numpy(full).unsqueeze(0) * I_AMP

    print(f'\nDirection {d_deg}°:')
    with torch.no_grad():
        V = net(x, dt=DT, show_progress=True)
    print(f'  V=[{V.min():.1f},{V.max():.1f}]')
    all_responses[d_deg] = V[0]

print('\nAll directions done.')

## 3. Direction Selectivity

In [ ]:
t_on_s = int(T_PRE / DT)
t_on_e = int((T_PRE + T_STIM) / DT)

dsi_types = ['R1-6', 'L1', 'Mi1', 'Mi9', 'Tm3',
             'T4a', 'T4b', 'T4c', 'T4d', 'T5a', 'T5b', 'T5c', 'T5d',
             'C2', 'C3', 'T1']

n_dir = len(directions)
print(f'{"Type":>8s}  {"DSI":>10s}  {"PD":>6s}')
for ct in dsi_types:
    idx = net.get_indices_by_type(ct)
    if len(idx) < 2: continue
    it = torch.tensor(idx)
    r_dir = []
    for d in directions:
        vd = all_responses[d]
        v_b = vd[:t_on_s, it].mean(dim=0)
        r = (vd[t_on_s:t_on_e, it] - v_b).abs().mean(dim=0)
        r_dir.append(r)
    r_dir = torch.stack(r_dir)
    r_max, pi = r_dir.max(dim=0)
    null_i = (pi + n_dir//2) % n_dir
    r_null = r_dir[null_i, torch.arange(len(idx))]
    dsi = (r_max - r_null) / (r_max + r_null + 1e-8)
    pd_rad = np.deg2rad([directions[i] for i in pi.numpy()])
    mean_pd = np.rad2deg(np.arctan2(np.sin(pd_rad).mean(), np.cos(pd_rad).mean())) % 360
    print(f'{ct:>8s}  {dsi.mean():.3f}±{dsi.std():.3f}  {mean_pd:5.0f}°')

## 4. LC neuron direction selectivity

In [ ]:
neurons_df = pd.read_csv(DATA_DIR + '/neurons.csv')
all_ids = np.sort(neurons_df['root_id'].unique())
id_to_idx = {int(r): i for i, r in enumerate(all_ids)}
neurons_df['idx'] = neurons_df['root_id'].map(id_to_idx)

lc_types = neurons_df[neurons_df['type'].str.startswith('LC', na=False)]['type'].value_counts()

print(f'{"LC Type":>10s}  {"n":>4s}  {"DSI":>10s}')
for ct in lc_types.head(10).index:
    il = neurons_df.loc[neurons_df['type']==ct, 'idx'].dropna().astype(int).tolist()
    if len(il) < 2: continue
    it = torch.tensor(il)
    r_dir = []
    for d in directions:
        vd = all_responses[d]
        v_b = vd[:t_on_s, it].mean(dim=0)
        r = (vd[t_on_s:t_on_e, it] - v_b).abs().mean(dim=0)
        r_dir.append(r)
    r_dir = torch.stack(r_dir)
    r_max, pi = r_dir.max(dim=0)
    null_i = (pi + n_dir//2) % n_dir
    r_null = r_dir[null_i, torch.arange(len(il))]
    dsi = (r_max - r_null) / (r_max + r_null + 1e-8)
    print(f'{ct:>10s}  {len(il):4d}  {dsi.mean():.3f}±{dsi.std():.3f}')